# Margin Analysis for QA Model

This notebook inspects token-level probability margins collected by `generate_answers_margin.py` and compares statistics between **correct** and **incorrect** answers.

- **Original metrics** (all tokens): `mean_margin`, `min_margin`, etc.
- **Content-only metrics**: same stats computed after excluding punctuation, whitespace, and common stopwords (to reduce inflation from high-margin stopwords).

Steps: load run → collect margins → optionally compute content-only stats → plot distributions and AUROC.

In [ ]:
import os
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Make plots render inline
%matplotlib inline

# Root of this repo (adjust if you run from a different cwd)
REPO_ROOT = Path("/home/xiaoniu/semantic_uncertainty")

# W&B run dirs can be in two places depending on where you ran the script:
# - REPO_ROOT/xiaoniu/uncertainty/wandb/...  (e.g. when cwd was REPO_ROOT)
# - REPO_ROOT/semantic_uncertainty/xiaoniu/uncertainty/wandb/...  (when cwd was REPO_ROOT/semantic_uncertainty)
WANDB_CANDIDATES = [
    REPO_ROOT / "xiaoniu" / "uncertainty" / "wandb",
    REPO_ROOT / "semantic_uncertainty" / "xiaoniu" / "uncertainty" / "wandb",
]

# Collect all run "files" dirs and pick the latest by modification time.
run_dirs = []
for root in WANDB_CANDIDATES:
    if root.exists():
        run_dirs.extend([p for p in root.glob("**/files") if p.is_dir()])
run_dirs = sorted(run_dirs, key=lambda p: p.stat().st_mtime, reverse=True)

if not run_dirs:
    raise RuntimeError("No W&B run 'files' directories found under any candidate path.\n"
                       "Make sure you have run generate_answers_margin.py with --single_answer_mode first.")

RUN_DIR = run_dirs[0]
print(f"Using RUN_DIR = {RUN_DIR}")

val_path = RUN_DIR / "validation_generations.pkl"
if not val_path.exists():
    raise FileNotFoundError(f"validation_generations.pkl not found at {val_path}")

with val_path.open("rb") as f:
    validation_generations = pickle.load(f)

# Read model name from run metadata so tokenizer and content-only logic match the run.
exp_path = RUN_DIR / "experiment_details.pkl"
if exp_path.exists():
    with exp_path.open("rb") as f:
        exp_details = pickle.load(f)
    args = exp_details.get("args")
    MODEL_NAME = getattr(args, "model_name", "Qwen/Qwen3-8B") if args is not None else "Qwen/Qwen3-8B"
else:
    MODEL_NAME = "Qwen/Qwen3-8B"
print(f"Model name from run: {MODEL_NAME}")

len(validation_generations)

Using RUN_DIR = /home/xiaoniu/semantic_uncertainty/semantic_uncertainty/xiaoniu/uncertainty/wandb/offline-run-20260319_053414-58g7qtqw/files
Model name from run: Qwen/Qwen3-8B


200

In [ ]:
# Collect mean_margin for correct vs incorrect examples.

correct_margins = []
incorrect_margins = []
examples_for_debug = []  # store a few examples for manual inspection

for ex_id, ex in validation_generations.items():
    mla = ex.get("most_likely_answer")
    if mla is None:
        continue

    acc = mla.get("accuracy")
    ms = mla.get("margin_stats")
    if ms is None:
        # This can happen if the run was produced without --single_answer_mode.
        continue

    mean_margin = ms.get("mean_margin")
    if mean_margin is None:
        continue

    if acc is None:
        # If accuracy is missing for some reason, skip.
        continue

    # Length consistency sanity check.
    token_ids = ms.get("generated_token_ids", [])
    margins = ms.get("margins", [])
    top2_per_step = ms.get("top2_per_step", [])
    log_liks = mla.get("token_log_likelihoods", [])

    if not (
        len(token_ids) == len(margins) == len(top2_per_step) == len(log_liks)
    ):
        print(f"[WARN] length mismatch for example {ex_id}: ",
              len(token_ids), len(margins), len(top2_per_step), len(log_liks))
        continue

    # Margin range sanity check.
    if not all(0.0 <= m <= 1.0 for m in margins):
        print(f"[WARN] margin out of [0,1] for example {ex_id}")
        continue

    if acc >= 0.5:
        correct_margins.append(mean_margin)
    else:
        incorrect_margins.append(mean_margin)

    if len(examples_for_debug) < 5:
        examples_for_debug.append((ex_id, ex))

print(f"#correct = {len(correct_margins)}, #incorrect = {len(incorrect_margins)}")

#correct = 106, #incorrect = 94


In [ ]:

    acc = mla.get("accuracy")
    ms = mla.get("margin_stats")
    if ms is None:
        # This can happen if the run was produced without --single_answer_mode.
        continue

    mean_margin = ms.get("mean_margin")
    if mean_margin is None:
        continue

    if acc is None:
        # If accuracy is missing for some reason, skip.
        continue

    # Length consistency sanity check.
    token_ids = ms.get("generated_token_ids", [])
    margins = ms.get("margins", [])
    top2_per_step = ms.get("top2_per_step", [])
    log_liks = mla.get("token_log_likelihoods", [])

    if not (
        len(token_ids) == len(margins) == len(top2_per_step) == len(log_liks)
    ):
        print(f"[WARN] length mismatch for example {ex_id}: ",
              len(token_ids), len(margins), len(top2_per_step), len(log_liks))
        continue

    # Margin range sanity check.
    if not all(0.0 <= m <= 1.0 for m in margins):
        print(f"[WARN] margin out of [0,1] for example {ex_id}")
        continue

    if acc >= 0.5:
        correct_margins.append(mean_margin)
    else:
        incorrect_margins.append(mean_margin)

    if len(examples_for_debug) < 5:
        examples_for_debug.append((ex_id, ex))

print(f"#correct = {len(correct_margins)}, #incorrect = {len(incorrect_margins)}")

In [6]:
# Content-only margin: define tokenizer and which tokens to exclude (analysis-side only).
# Generation pipeline and saved fields (margins, mean_margin, ...) are unchanged.

import string
from transformers import AutoTokenizer

# MODEL_NAME set in Cell 1 from run metadata (experiment_details.pkl); fallback Qwen/Qwen3-8B
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Simple English stopword list (content-only excludes these)
STOPWORDS = {
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "must", "can", "to", "of", "in", "for", "on",
    "with", "at", "by", "from", "as", "it", "its", "or", "and", "that",
    "this", "but", "if", "then", "so", "when", "than", "no", "not",
}

PUNCTUATION = set(string.punctuation)

def is_content_token(token_str: str) -> bool:
    """True if token should count toward content-only margin (exclude punct, space, stopwords)."""
    s = token_str.strip()
    if not s:
        return False
    if all(c in PUNCTUATION or c.isspace() for c in s):
        return False
    if s.lower() in STOPWORDS:
        return False
    return True

In [ ]:
# Compute content-only margin stats per sample (analysis-side only).
# Original margin_stats fields are unchanged; we add derived lists for comparison.

samples_for_auc = []  # list of dicts: accuracy, mean_margin, mean_margin_content, min_margin, min_margin_content
correct_mean_content = []
incorrect_mean_content = []
correct_min_content = []
incorrect_min_content = []

for ex_id, ex in validation_generations.items():
    mla = ex.get("most_likely_answer")
    if mla is None:
        continue
    ms = mla.get("margin_stats")
    if ms is None:
        continue
    acc = mla.get("accuracy")
    if acc is None:
        continue

    token_ids = ms.get("generated_token_ids", [])
    margins = ms.get("margins", [])
    if len(token_ids) != len(margins):
        continue

    # Which positions are content tokens (exclude punct, whitespace, stopwords)
    content_token_indices = []
    for i, tid in enumerate(token_ids):
        tok_str = tokenizer.decode([tid], skip_special_tokens=False).strip()
        if is_content_token(tok_str):
            content_token_indices.append(i)

    content_margins = [margins[i] for i in content_token_indices]

    # Content-only aggregates (nan if no content tokens)
    if content_margins:
        mean_margin_content = float(np.mean(content_margins))
        min_margin_content = float(np.min(content_margins))
        max_margin_content = float(np.max(content_margins))
        low_margin_ratio_content_0_1 = float(np.mean(np.array(content_margins) < 0.1))
    else:
        mean_margin_content = min_margin_content = max_margin_content = low_margin_ratio_content_0_1 = np.nan

    mean_margin = ms.get("mean_margin")
    min_margin = ms.get("min_margin")
    if mean_margin is None or min_margin is None:
        continue

    samples_for_auc.append({
        "accuracy": acc,
        "mean_margin": mean_margin,
        "mean_margin_content": mean_margin_content,
        "min_margin": min_margin,
        "min_margin_content": min_margin_content,
    })

    if not np.isnan(mean_margin_content):
        if acc >= 0.5:
            correct_mean_content.append(mean_margin_content)
            correct_min_content.append(min_margin_content)
        else:
            incorrect_mean_content.append(mean_margin_content)
            incorrect_min_content.append(min_margin_content)

print(f"Content-only: #correct = {len(correct_mean_content)}, #incorrect = {len(incorrect_mean_content)}")
print(f"Samples for AUROC: {len(samples_for_auc)}")

In [ ]:
# Plot distributions: original mean_margin and content-only mean_margin_content.

bins = np.linspace(0.0, 1.0, 30)

# Original mean_margin
plt.figure(figsize=(6, 4))
plt.hist(correct_margins, bins=bins, alpha=0.5, label="correct", density=True)
plt.hist(incorrect_margins, bins=bins, alpha=0.5, label="incorrect", density=True)
plt.xlabel("mean_margin")
plt.ylabel("density")
plt.title("Distribution of mean_margin (all tokens): correct vs incorrect")
plt.legend()
plt.tight_layout()
plt.show()

# Content-only mean_margin_content
if correct_mean_content and incorrect_mean_content:
    plt.figure(figsize=(6, 4))
    plt.hist(correct_mean_content, bins=bins, alpha=0.5, label="correct", density=True)
    plt.hist(incorrect_mean_content, bins=bins, alpha=0.5, label="incorrect", density=True)
    plt.xlabel("mean_margin_content")
    plt.ylabel("density")
    plt.title("Distribution of mean_margin_content (excl. punct/space/stopwords): correct vs incorrect")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# AUROC: compare original vs content-only (mean_margin and min_margin).
# Label: correct=1, incorrect=0. Higher margin => more confident => more likely correct.

from sklearn.metrics import roc_auc_score, roc_curve

# Filter to samples with valid content-only stats for fair comparison
valid = [s for s in samples_for_auc if not np.isnan(s["mean_margin_content"])]
if not valid:
    print("No samples with content-only stats; run content-only cell first.")
else:
    y = np.array([1 if s["accuracy"] >= 0.5 else 0 for s in valid])

    for name, key in [
        ("mean_margin", "mean_margin"),
        ("mean_margin_content", "mean_margin_content"),
        ("min_margin", "min_margin"),
        ("min_margin_content", "min_margin_content"),
    ]:
        scores = np.array([s[key] for s in valid])
        # Drop any nan for this metric
        mask = ~np.isnan(scores)
        if mask.sum() < 2:
            print(f"{name}: not enough valid scores")
            continue
        y_m, s_m = y[mask], scores[mask]
        auroc = roc_auc_score(y_m, s_m)
        print(f"AUROC ({name}): {auroc:.4f}")

    # Plot ROC curves for mean_margin vs mean_margin_content
    scores_full = np.array([s["mean_margin"] for s in valid])
    scores_content = np.array([s["mean_margin_content"] for s in valid])
    fpr_full, tpr_full, _ = roc_curve(y, scores_full)
    fpr_content, tpr_content, _ = roc_curve(y, scores_content)

    plt.figure(figsize=(5, 5))
    plt.plot(fpr_full, tpr_full, label=f"mean_margin (AUC={roc_auc_score(y, scores_full):.3f})")
    plt.plot(fpr_content, tpr_content, label=f"mean_margin_content (AUC={roc_auc_score(y, scores_content):.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("ROC: mean_margin vs mean_margin_content")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect one example: token text + logprob + margin (to check if stopwords inflate margin).

ex_id, ex = examples_for_debug[0]
mla = ex["most_likely_answer"]
ms = mla["margin_stats"]

print("Example id:", ex_id)
print("Question:", ex.get("question"))
print("Answer:", mla["response"])
print("Accuracy:", mla["accuracy"])
print("mean_margin:", ms["mean_margin"])
print("answer_length:", ms["answer_length"])
print()

# tokenizer from Cell 3 (same model as run)
for idx, (tok_id, logp, step) in enumerate(
    zip(ms["generated_token_ids"], mla["token_log_likelihoods"], ms["top2_per_step"])
):
    tok_text = tokenizer.decode([tok_id], skip_special_tokens=False)
    print(
        f"step={idx:02d}, token_text={repr(tok_text)}, token_id={tok_id}, logp={logp:.4f}, "
        f"top1_p={step['top1_prob']:.4f}, top2_p={step['top2_prob']:.4f}, margin={step['margin']:.4f}"
    )

In [ ]:
<!-- intentionally left empty -->